# Notebook 01 — Data Acquisition

**Purpose:** Download all raw data, verify integrity, document provenance.

**Outputs:**
- `data/raw/` — all downloaded files
- `data/raw/MANIFEST.json` — provenance log (URLs, checksums, descriptions)

**Data sources:**
1. USGS SAR-HiSAR Global Mosaic (351 m/pixel, ~1 GB GeoTIFF)
2. NLDSAR Denoised Dataset (Zenodo)
3. Lopes et al. (2020) Geomorphological Map (label ground truth)
4. BIDR swaths (subset covering Dragonfly landing area)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src') if os.path.basename(os.getcwd()) == 'notebooks' else 'src')
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else '.')

import json
import hashlib
import datetime
from pathlib import Path

import requests
from tqdm.notebook import tqdm

from src.utils import RAW_DIR, sha256_file, save_manifest, load_manifest, get_logger

log = get_logger('01_data_acquisition')
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f'Raw data directory: {RAW_DIR}')

## Helper: resumable download

In [ ]:
def download_file(url, dest_path, description="", chunk_size=1 << 20):
    """Download a file with progress bar and resume support."""
    dest_path = Path(dest_path)
    if dest_path.exists():
        log.info(f'Already exists: {dest_path.name} ({dest_path.stat().st_size:,} bytes)')
        return dest_path

    log.info(f'Downloading: {description or url}')
    headers = {}
    partial = dest_path.with_suffix(dest_path.suffix + '.partial')
    if partial.exists():
        resume_pos = partial.stat().st_size
        headers['Range'] = f'bytes={resume_pos}-'
        log.info(f'Resuming from {resume_pos:,} bytes')
    else:
        resume_pos = 0

    resp = requests.get(url, headers=headers, stream=True, timeout=60)
    resp.raise_for_status()

    total = int(resp.headers.get('content-length', 0)) + resume_pos
    mode = 'ab' if resume_pos else 'wb'

    with open(partial, mode) as f, tqdm(
        total=total, initial=resume_pos, unit='B', unit_scale=True,
        desc=dest_path.name
    ) as pbar:
        for chunk in resp.iter_content(chunk_size=chunk_size):
            f.write(chunk)
            pbar.update(len(chunk))

    partial.rename(dest_path)
    log.info(f'Saved: {dest_path.name} ({dest_path.stat().st_size:,} bytes)')
    return dest_path

## 1. USGS SAR-HiSAR Global Mosaic

Single GeoTIFF, ~1 GB, 351 m/pixel, Ku-band backscatter (σ₀), grayscale.

In [ ]:
# The USGS Astrogeology page may redirect. We try the direct download URL.
# If this fails, manually download from:
# https://astrogeology.usgs.gov/search/map/titan_cassini_sar_hisar_global_mosaic_351m

SAR_MOSAIC_URL = (
    "https://planetarymaps.usgs.gov/mosaic/Titan/"
    "Titan_SAR_HiSAR_Global_Mosaic_351m.tif"
)
SAR_MOSAIC_FILENAME = "Titan_SAR_HiSAR_Global_Mosaic_351m.tif"

sar_mosaic_path = RAW_DIR / SAR_MOSAIC_FILENAME

try:
    download_file(
        SAR_MOSAIC_URL,
        sar_mosaic_path,
        description="USGS SAR-HiSAR Global Mosaic (351 m/pixel)"
    )
except Exception as e:
    log.warning(f"Direct download failed: {e}")
    log.warning("Try alternative URLs or manual download from USGS Astrogeology.")
    # Alternative URL patterns to try
    alt_urls = [
        "https://astropedia.astrogeology.usgs.gov/download/Titan/Cassini/SAR/Titan_SAR_HiSAR_Global_Mosaic_351m.tif",
    ]
    for alt_url in alt_urls:
        try:
            log.info(f"Trying: {alt_url}")
            download_file(alt_url, sar_mosaic_path, description="SAR Mosaic (alt URL)")
            SAR_MOSAIC_URL = alt_url
            break
        except Exception as e2:
            log.warning(f"Also failed: {e2}")
    else:
        log.error(
            "Could not auto-download SAR mosaic. Please download manually from:\n"
            "https://astrogeology.usgs.gov/search/map/titan_cassini_sar_hisar_global_mosaic_351m\n"
            f"Save to: {sar_mosaic_path}"
        )

In [ ]:
# Verify the SAR mosaic opens correctly
import rasterio

if sar_mosaic_path.exists():
    with rasterio.open(sar_mosaic_path) as src:
        print(f'CRS:        {src.crs}')
        print(f'Resolution: {src.res}')
        print(f'Shape:      {src.height} x {src.width}')
        print(f'Bounds:     {src.bounds}')
        print(f'Dtype:      {src.dtypes}')
        print(f'NoData:     {src.nodata}')
else:
    print('SAR mosaic not yet downloaded — see above.')

## 2. NLDSAR Denoised Dataset

Non-local means denoised SAR. Available on Zenodo: https://zenodo.org/records/528545

In [ ]:
# The Zenodo record may contain multiple files. We'll fetch the record metadata
# first to discover the actual download links.

NLDSAR_ZENODO_RECORD = "528545"
NLDSAR_API_URL = f"https://zenodo.org/api/records/{NLDSAR_ZENODO_RECORD}"

nldsar_dir = RAW_DIR / "nldsar"
nldsar_dir.mkdir(exist_ok=True)

try:
    resp = requests.get(NLDSAR_API_URL, timeout=30)
    resp.raise_for_status()
    record = resp.json()
    
    print(f"Title: {record['metadata']['title']}")
    print(f"DOI:   {record['doi']}")
    print(f"Files: {len(record['files'])}")
    print()
    
    nldsar_files = []
    for f in record['files']:
        size_mb = f['size'] / 1e6
        print(f"  {f['key']:40s}  {size_mb:8.1f} MB")
        nldsar_files.append({
            'filename': f['key'],
            'url': f['links']['self'],
            'size': f['size'],
            'checksum': f['checksum'],  # md5:...
        })
except Exception as e:
    log.warning(f"Could not fetch Zenodo record: {e}")
    nldsar_files = []

In [ ]:
# Download NLDSAR files (these can be large — download selectively if needed)
for finfo in nldsar_files:
    dest = nldsar_dir / finfo['filename']
    try:
        download_file(finfo['url'], dest, description=f"NLDSAR: {finfo['filename']}")
    except Exception as e:
        log.warning(f"Failed to download {finfo['filename']}: {e}")

## 3. Lopes et al. (2020) Geomorphological Map

This is the **single most important file** — it provides ground-truth training labels.

Six classes: plains, dunes, hummocky/mountainous, lakes/seas, labyrinth, craters.

**Search strategy:**
1. Check USGS Astrogeology for Titan geomorphological map products
2. Check the Nature Astronomy supplementary materials (doi:10.1038/s41550-019-0917-6)
3. Check public data repositories (Zenodo, Figshare, etc.)
4. If unavailable as raster, plan to digitise from published figure

In [ ]:
# Strategy 1: Check USGS Astrogeology for Titan geomorphological map
# Known USGS product pages for Titan geological maps:

GEOMORPH_CANDIDATES = [
    {
        'name': 'USGS Titan Global Geologic Map (SIM-3414)',
        'url': 'https://astropedia.astrogeology.usgs.gov/download/Titan/Geology/Titan_global_geology_GIS.zip',
        'info': 'https://astrogeology.usgs.gov/search/map/Titan/Geology/Titan_global_geology',
        'type': 'shapefile',
    },
    {
        'name': 'Lopes et al. 2020 supplementary (Nature Astronomy)',
        'url': None,  # Need to check manually
        'info': 'https://doi.org/10.1038/s41550-019-0917-6',
        'type': 'unknown',
    },
]

geomorph_dir = RAW_DIR / 'geomorphology'
geomorph_dir.mkdir(exist_ok=True)

for candidate in GEOMORPH_CANDIDATES:
    print(f"\n{'='*60}")
    print(f"Source: {candidate['name']}")
    print(f"Info:   {candidate['info']}")
    if candidate['url']:
        dest = geomorph_dir / Path(candidate['url']).name
        try:
            download_file(candidate['url'], dest, description=candidate['name'])
            print(f"Downloaded: {dest}")
        except Exception as e:
            log.warning(f"Failed: {e}")
    else:
        print(f"No direct URL — manual check required at {candidate['info']}")

In [ ]:
# If we downloaded the USGS GIS zip, inspect its contents
import zipfile

gis_zip = geomorph_dir / 'Titan_global_geology_GIS.zip'
if gis_zip.exists():
    with zipfile.ZipFile(gis_zip) as zf:
        print(f"Contents of {gis_zip.name}:")
        for info in zf.infolist():
            print(f"  {info.filename:50s}  {info.file_size:>10,} bytes")
        
        # Extract
        extract_dir = geomorph_dir / 'usgs_geology'
        zf.extractall(extract_dir)
        print(f"\nExtracted to: {extract_dir}")
else:
    print("USGS GIS zip not available — trying alternative approaches.")

In [ ]:
# Inspect any shapefiles we found
import glob

shapefiles = list(geomorph_dir.rglob('*.shp'))
if shapefiles:
    import fiona
    for shp in shapefiles:
        print(f"\nShapefile: {shp.name}")
        with fiona.open(shp) as src:
            print(f"  CRS:      {src.crs}")
            print(f"  Schema:   {src.schema}")
            print(f"  Features: {len(src)}")
            # Show unique values in classification field
            classes = set()
            for feat in src:
                props = feat['properties']
                # Look for terrain/geology classification field
                for key, val in props.items():
                    if any(kw in key.lower() for kw in ['class', 'type', 'unit', 'geol', 'morph', 'terrain']):
                        classes.add((key, val))
            if classes:
                print(f"  Classification fields found:")
                by_field = {}
                for k, v in classes:
                    by_field.setdefault(k, set()).add(v)
                for k, vals in by_field.items():
                    print(f"    {k}: {sorted(vals)}")
else:
    print("No shapefiles found. The geomorphological map may need to be:")
    print("  1. Requested from the authors (Lopes et al.)")
    print("  2. Digitised from the published figure (6-colour map)")
    print("  3. Derived from SAR backscatter thresholds (partial solution)")

## 4. BIDR Swaths (Dragonfly Landing Region)

Individual SAR swaths at ~175 m/pixel. Targeting the Shangri-La / Selk Crater region
(approx. 0°–10°N, 160°–200°W).

BIDR products are on PDS: https://pds-imaging.jpl.nasa.gov/volumes/radar.html

In [ ]:
# Key Cassini flybys covering the Dragonfly landing region:
# T8 (CORADR_0048), T28 (CORADR_0101), T29 (CORADR_0102), 
# T41 (CORADR_0138), T83 (CORADR_0211), T95 (CORADR_0225)
# These have good SAR coverage of Shangri-La and Selk Crater.

PDS_BASE_URL = "https://pds-imaging.jpl.nasa.gov/data/cassini/cassini_orbiter/"

# We'll target a small subset to keep download manageable
BIDR_TARGETS = [
    {
        'volume': 'CORADR_0048',
        'flyby': 'T8',
        'description': 'T8 flyby — Shangri-La dune field coverage',
    },
    {
        'volume': 'CORADR_0211',
        'flyby': 'T83',
        'description': 'T83 flyby — Selk Crater region',
    },
]

bidr_dir = RAW_DIR / 'bidr'
bidr_dir.mkdir(exist_ok=True)

print("BIDR swath targets:")
for t in BIDR_TARGETS:
    print(f"  {t['volume']} ({t['flyby']}): {t['description']}")

print("\nNote: BIDR volumes are large. The download cells below will attempt")
print("to fetch only the BIDR S01 (primary beam) .IMG files.")
print("If automatic download fails, use the PDS Imaging Atlas:")
print("https://pds-imaging.jpl.nasa.gov/volumes/radar.html")

In [ ]:
# Attempt to list and download BIDR products from PDS
# PDS directory structure: CORADR_XXXX/DATA/BIDR/

for target in BIDR_TARGETS:
    vol = target['volume']
    vol_dir = bidr_dir / vol
    vol_dir.mkdir(exist_ok=True)
    
    # Try to fetch the BIDR directory listing
    bidr_url = f"{PDS_BASE_URL}{vol}/DATA/BIDR/"
    print(f"\nChecking {bidr_url} ...")
    
    try:
        resp = requests.get(bidr_url, timeout=30)
        resp.raise_for_status()
        
        # Parse the directory listing for S01 BIDR .IMG files
        import re
        img_files = re.findall(r'href="(BIBQ.*S01[^"]*\.IMG)"', resp.text, re.IGNORECASE)
        if not img_files:
            # Try broader pattern
            img_files = re.findall(r'href="(BI[^"]*\.IMG)"', resp.text, re.IGNORECASE)
        
        print(f"  Found {len(img_files)} .IMG files")
        for fname in img_files[:5]:  # Show first 5
            print(f"    {fname}")
        if len(img_files) > 5:
            print(f"    ... and {len(img_files) - 5} more")
        
        # Download the first S01 primary beam file (usually the main SAR image)
        s01_files = [f for f in img_files if 'S01' in f.upper()]
        if s01_files:
            for fname in s01_files[:1]:  # Just the first one to start
                file_url = f"{bidr_url}{fname}"
                dest = vol_dir / fname
                try:
                    download_file(file_url, dest, description=f"{vol} {fname}")
                except Exception as e:
                    log.warning(f"Failed to download {fname}: {e}")
        else:
            log.info(f"No S01 files found for {vol}. Available: {img_files[:3]}")
            
    except Exception as e:
        log.warning(f"Could not access {bidr_url}: {e}")
        log.info(f"Manual download required from PDS for {vol}")

## 5. Build Provenance Manifest

In [ ]:
# Build MANIFEST.json logging all downloaded files
manifest = {
    'created': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'description': 'Titan SAR terrain classification — raw data provenance',
    'files': []
}

# Walk data/raw/ and log every file
for fpath in sorted(RAW_DIR.rglob('*')):
    if fpath.is_file() and fpath.name != 'MANIFEST.json':
        rel_path = str(fpath.relative_to(RAW_DIR))
        size = fpath.stat().st_size
        
        # Determine source URL
        source_url = 'unknown'
        description = ''
        if 'SAR_HiSAR' in fpath.name:
            source_url = SAR_MOSAIC_URL
            description = 'USGS SAR-HiSAR Global Mosaic, 351 m/pixel'
        elif 'nldsar' in str(rel_path).lower():
            source_url = f'https://zenodo.org/records/{NLDSAR_ZENODO_RECORD}'
            description = 'NLDSAR denoised SAR data (Lucas et al. 2014)'
        elif 'geomorph' in str(rel_path).lower() or 'geol' in str(rel_path).lower():
            source_url = 'https://astrogeology.usgs.gov'
            description = 'Geomorphological/geological map data'
        elif 'bidr' in str(rel_path).lower() or 'CORADR' in str(rel_path):
            source_url = PDS_BASE_URL
            description = 'Cassini BIDR SAR swath'
        
        # Compute checksum (skip for very large files > 2GB)
        if size < 2e9:
            checksum = sha256_file(fpath)
        else:
            checksum = 'skipped-large-file'
        
        manifest['files'].append({
            'path': rel_path,
            'size_bytes': size,
            'sha256': checksum,
            'source_url': source_url,
            'description': description,
            'access_date': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d'),
        })

save_manifest(manifest)
print(f"Manifest saved with {len(manifest['files'])} files.")
for f in manifest['files']:
    print(f"  {f['path']:50s}  {f['size_bytes']:>12,} bytes")

## 6. Verification Summary

In [ ]:
print("=" * 60)
print("DATA ACQUISITION SUMMARY")
print("=" * 60)

checks = {
    'SAR Mosaic': sar_mosaic_path.exists(),
    'NLDSAR Data': any(nldsar_dir.iterdir()) if nldsar_dir.exists() else False,
    'Geomorphological Map': any(geomorph_dir.rglob('*.shp')) or any(geomorph_dir.rglob('*.tif')),
    'BIDR Swaths': any(bidr_dir.rglob('*.IMG')) if bidr_dir.exists() else False,
}

for item, ok in checks.items():
    status = 'OK' if ok else 'MISSING'
    print(f"  [{status:7s}] {item}")

if not all(checks.values()):
    print("\nSome data sources need manual intervention — see notes above.")
    print("The geomorphological map is the CRITICAL dependency.")
    print("Proceed to Notebook 02 once at least SAR mosaic + label map are available.")
else:
    print("\nAll data acquired. Ready for Notebook 02.")